In [1]:
%pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sqlite3
import sqlalchemy

from pathlib import Path
from datetime import datetime

print("SQLite version:", sqlite3.sqlite_version)
print("SQLAlchemy version:", sqlalchemy.__version__)
print("Database environment ready!")

SQLite version: 3.49.1
SQLAlchemy version: 2.1.9
Database environment ready!


In [3]:
PROJECT_ROOT = Path.cwd().parent

DATABASE_PATH = PROJECT_ROOT / "database"

DATABASE_PATH.mkdir(
    parents=True,
    exist_ok=True
)

DB_FILE = DATABASE_PATH / "skillbridge.db"

print("Project:", PROJECT_ROOT)
print("Database folder:", DATABASE_PATH)
print("Database file:", DB_FILE)

Project: /Users/rizwansalim/Desktop/SkillBridgeByZequa
Database folder: /Users/rizwansalim/Desktop/SkillBridgeByZequa/database
Database file: /Users/rizwansalim/Desktop/SkillBridgeByZequa/database/skillbridge.db


In [4]:
print("DAY 14 TASK 1 VALIDATION")
print("-" * 40)

checks = {
    "Project directory": PROJECT_ROOT.exists(),
    "Database directory": DATABASE_PATH.exists(),
    "SQLite available": sqlite3.sqlite_version is not None,
    "SQLAlchemy available": sqlalchemy.__version__ is not None
}

for name, status in checks.items():
    print(
        ("FOUND" if status else "MISSING"),
        "|",
        name
    )

print("-" * 40)
print("Validation:", "PASSED" if all(checks.values()) else "FAILED")

DAY 14 TASK 1 VALIDATION
----------------------------------------
FOUND | Project directory
FOUND | Database directory
FOUND | SQLite available
FOUND | SQLAlchemy available
----------------------------------------
Validation: PASSED


In [5]:
from sqlalchemy import create_engine
from sqlalchemy.orm import declarative_base, sessionmaker

print("SQLAlchemy components imported!")

SQLAlchemy components imported!


In [6]:
DATABASE_URL = f"sqlite:///{DB_FILE.as_posix()}"

print("Database URL:", DATABASE_URL)

Database URL: sqlite:////Users/rizwansalim/Desktop/SkillBridgeByZequa/database/skillbridge.db


In [7]:
engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False},
    echo=False
)

print("Database engine created!")

Database engine created!


In [8]:
SessionLocal = sessionmaker(
    autocommit=False,
    autoflush=False,
    bind=engine
)

Base = declarative_base()

print("Database session configured!")
print("SQLAlchemy Base configured!")

Database session configured!
SQLAlchemy Base configured!


In [9]:
from sqlalchemy import text

try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT 1"))
        print("Database connection:", result.scalar())

except Exception as error:
    print("Database connection failed:", error)

Database connection: 1


In [10]:
print("DAY 14 TASK 2 VALIDATION")
print("-" * 40)

print("Engine created:", engine is not None)
print("Session configured:", SessionLocal is not None)
print("Base configured:", Base is not None)
print("Database connection:", DB_FILE.parent.exists())

print("-" * 40)

DAY 14 TASK 2 VALIDATION
----------------------------------------
Engine created: True
Session configured: True
Base configured: True
Database connection: True
----------------------------------------


In [11]:
database_code = '''
from pathlib import Path
from sqlalchemy import create_engine
from sqlalchemy.orm import declarative_base, sessionmaker

PROJECT_ROOT = Path(__file__).resolve().parent.parent

DATABASE_PATH = PROJECT_ROOT / "database"
DATABASE_PATH.mkdir(parents=True, exist_ok=True)

DB_FILE = DATABASE_PATH / "skillbridge.db"

DATABASE_URL = f"sqlite:///{DB_FILE.as_posix()}"

engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False},
    echo=False
)

SessionLocal = sessionmaker(
    autocommit=False,
    autoflush=False,
    bind=engine
)

Base = declarative_base()

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()
'''

database_module_path = PROJECT_ROOT / "backend" / "database.py"

database_module_path.write_text(
    database_code.strip(),
    encoding="utf-8"
)

print("Backend database module created!")

Backend database module created!


In [12]:
database_init_path = PROJECT_ROOT / "database" / "__init__.py"

database_init_path.write_text(
    "# SkillBridge database package",
    encoding="utf-8"
)

print("Database package initialized!")

Database package initialized!


In [13]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.database import (
    engine as backend_engine,
    SessionLocal as BackendSessionLocal,
    Base as BackendBase,
    get_db
)

print("Backend database integration successful!")

Backend database integration successful!


In [14]:
from sqlalchemy import text

with backend_engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))

    print("Database connection:", result.scalar())

print("Database file exists:", DB_FILE.exists())

Database connection: 1
Database file exists: True


In [15]:
print("DAY 14 TASK 3 VALIDATION")
print("-" * 45)

checks = {
    "Backend database.py": database_module_path.exists(),
    "Database package": database_init_path.exists(),
    "SQLAlchemy engine": backend_engine is not None,
    "Database session": BackendSessionLocal is not None,
    "Declarative Base": BackendBase is not None
}

for name, status in checks.items():
    print(("FOUND" if status else "MISSING"), "|", name)

print("-" * 45)
print("Validation:", "PASSED" if all(checks.values()) else "FAILED")

DAY 14 TASK 3 VALIDATION
---------------------------------------------
FOUND | Backend database.py
FOUND | Database package
FOUND | SQLAlchemy engine
FOUND | Database session
FOUND | Declarative Base
---------------------------------------------
Validation: PASSED


In [16]:
candidate_model_code = '''
from sqlalchemy import Column, Integer, String, Text, DateTime
from sqlalchemy.sql import func
from backend.database import Base

class Candidate(Base):

    __tablename__ = "candidates"

    id = Column(Integer, primary_key=True, index=True)

    full_name = Column(String(150), nullable=False)
    email = Column(String(150), unique=True, index=True)
    phone = Column(String(30))

    education = Column(Text)
    experience = Column(Text)
    location = Column(String(150))

    created_at = Column(
        DateTime(timezone=True),
        server_default=func.now()
    )

    updated_at = Column(
        DateTime(timezone=True),
        onupdate=func.now()
    )
'''

candidate_model_path = (
    PROJECT_ROOT / "backend" / "models" / "candidate.py"
)

candidate_model_path.parent.mkdir(parents=True, exist_ok=True)

candidate_model_path.write_text(
    candidate_model_code.strip(),
    encoding="utf-8"
)

print("Candidate model created!")

Candidate model created!


In [17]:
models_init_path = PROJECT_ROOT / "backend" / "models" / "__init__.py"

models_init_path.write_text(
    "# SkillBridge SQLAlchemy models",
    encoding="utf-8"
)

print("Models package initialized!")

Models package initialized!


In [18]:
import importlib

import backend.models.candidate


from backend.models.candidate import Candidate

print("Model:", Candidate.__name__)
print("Table:", Candidate.__tablename__)
print("Columns:", list(Candidate.__table__.columns.keys()))

Model: Candidate
Table: candidates
Columns: ['id', 'full_name', 'email', 'phone', 'education', 'experience', 'location', 'created_at', 'updated_at']


In [19]:
from backend.database import Base, engine

Base.metadata.create_all(bind=engine)

print("Candidate table created successfully!")

Candidate table created successfully!


In [20]:
from sqlalchemy import inspect

inspector = inspect(engine)

tables = inspector.get_table_names()

print("Candidate table exists:", "candidates" in tables)

if "candidates" in tables:
    print("Columns:")
    for column in inspector.get_columns("candidates"):
        print(column["name"], "|", column["type"])

Candidate table exists: True
Columns:
id | INTEGER
full_name | VARCHAR(150)
email | VARCHAR(150)
phone | VARCHAR(30)
education | TEXT
experience | TEXT
location | VARCHAR(150)
created_at | DATETIME
updated_at | DATETIME


In [21]:
from sqlalchemy import inspect
from backend.database import engine

inspector = inspect(engine)
tables = inspector.get_table_names()

print("DAY 14 TASK 4 VALIDATION")
print("-" * 40)
print("Candidate table:", "candidates" in tables)

if "candidates" in tables:
    print("Columns:")
    for column in inspector.get_columns("candidates"):
        print("-", column["name"])

print("-" * 40)

DAY 14 TASK 4 VALIDATION
----------------------------------------
Candidate table: True
Columns:
- id
- full_name
- email
- phone
- education
- experience
- location
- created_at
- updated_at
----------------------------------------


In [22]:
resume_model_code = '''
from sqlalchemy import Column, Integer, String, DateTime, ForeignKey
from sqlalchemy.sql import func
from backend.database import Base

class Resume(Base):

    __tablename__ = "resumes"

    id = Column(Integer, primary_key=True, index=True)

    candidate_id = Column(
        Integer,
        ForeignKey("candidates.id"),
        nullable=False
    )

    filename = Column(String(255), nullable=False)
    file_path = Column(String(500), nullable=False)
    file_type = Column(String(20))
    file_size = Column(Integer)

    uploaded_at = Column(
        DateTime(timezone=True),
        server_default=func.now()
    )
'''

resume_model_path = PROJECT_ROOT / "backend" / "models" / "resume.py"

resume_model_path.write_text(
    resume_model_code.strip(),
    encoding="utf-8"
)

print("Resume model created!")

Resume model created!


In [23]:
from backend.models.resume import Resume

print("Model:", Resume.__name__)
print("Table:", Resume.__tablename__)
print("Columns:", list(Resume.__table__.columns.keys()))

Model: Resume
Table: resumes
Columns: ['id', 'candidate_id', 'filename', 'file_path', 'file_type', 'file_size', 'uploaded_at']


In [24]:
from backend.models.resume import Resume

print("Model:", Resume.__name__)
print("Table:", Resume.__tablename__)
print("Columns:", list(Resume.__table__.columns.keys()))

Model: Resume
Table: resumes
Columns: ['id', 'candidate_id', 'filename', 'file_path', 'file_type', 'file_size', 'uploaded_at']


In [25]:
from backend.models.resume import Resume

print("Model:", Resume.__name__)
print("Table:", Resume.__tablename__)
print("Columns:", list(Resume.__table__.columns.keys()))

Model: Resume
Table: resumes
Columns: ['id', 'candidate_id', 'filename', 'file_path', 'file_type', 'file_size', 'uploaded_at']


In [26]:
from backend.database import Base, engine

Base.metadata.create_all(bind=engine)

print("Resume table created successfully!")

Resume table created successfully!


In [27]:
from sqlalchemy import inspect

inspector = inspect(engine)

tables = inspector.get_table_names()

print("DAY 14 TASK 5 VALIDATION")
print("-" * 40)

print("Resume table exists:", "resumes" in tables)

if "resumes" in tables:
    for column in inspector.get_columns("resumes"):
        print("-", column["name"])

print("-" * 40)

DAY 14 TASK 5 VALIDATION
----------------------------------------
Resume table exists: True
- id
- candidate_id
- filename
- file_path
- file_type
- file_size
- uploaded_at
----------------------------------------


In [28]:
from sqlalchemy import inspect
from backend.database import engine

inspector = inspect(engine)
tables = inspector.get_table_names()

print("DAY 14 TASK 5 VALIDATION")
print("-" * 40)

print("Resume table exists:", "resumes" in tables)

if "resumes" in tables:
    print("Columns:")
    for column in inspector.get_columns("resumes"):
        print("-", column["name"])

    foreign_keys = inspector.get_foreign_keys("resumes")
    print("Foreign keys:", foreign_keys)

print("-" * 40)

DAY 14 TASK 5 VALIDATION
----------------------------------------
Resume table exists: True
Columns:
- id
- candidate_id
- filename
- file_path
- file_type
- file_size
- uploaded_at
Foreign keys: [{'name': None, 'constrained_columns': ['candidate_id'], 'referred_schema': None, 'referred_table': 'candidates', 'referred_columns': ['id'], 'options': {}}]
----------------------------------------


In [29]:
skill_model_code = '''
from sqlalchemy import Column, Integer, String, ForeignKey, DateTime
from sqlalchemy.sql import func
from backend.database import Base

class Skill(Base):

    __tablename__ = "skills"

    id = Column(Integer, primary_key=True, index=True)

    candidate_id = Column(
        Integer,
        ForeignKey("candidates.id"),
        nullable=False
    )

    skill_name = Column(String(150), nullable=False)
    proficiency_level = Column(String(50))
    years_of_experience = Column(Integer)

    created_at = Column(
        DateTime(timezone=True),
        server_default=func.now()
    )
'''

skill_model_path = PROJECT_ROOT / "backend" / "models" / "skill.py"

skill_model_path.write_text(
    skill_model_code.strip(),
    encoding="utf-8"
)

print("Skill model created!")

Skill model created!


In [30]:
from backend.models.skill import Skill

print("Model:", Skill.__name__)
print("Table:", Skill.__tablename__)
print("Columns:", list(Skill.__table__.columns.keys()))

Model: Skill
Table: skills
Columns: ['id', 'candidate_id', 'skill_name', 'proficiency_level', 'years_of_experience', 'created_at']


In [31]:
from backend.database import Base, engine

Base.metadata.create_all(bind=engine)

print("Skills table created successfully!")

Skills table created successfully!


In [32]:
from sqlalchemy import inspect

inspector = inspect(engine)
tables = inspector.get_table_names()

print("DAY 14 TASK 6 VALIDATION")
print("-" * 40)

print("Skills table exists:", "skills" in tables)

if "skills" in tables:
    for column in inspector.get_columns("skills"):
        print("-", column["name"])

    print("Foreign keys:", inspector.get_foreign_keys("skills"))

print("-" * 40)

DAY 14 TASK 6 VALIDATION
----------------------------------------
Skills table exists: True
- id
- candidate_id
- skill_name
- proficiency_level
- years_of_experience
- created_at
Foreign keys: [{'name': None, 'constrained_columns': ['candidate_id'], 'referred_schema': None, 'referred_table': 'candidates', 'referred_columns': ['id'], 'options': {}}]
----------------------------------------


In [33]:
recommendation_model_code = '''
from sqlalchemy import Column, Integer, String, Float, Text, ForeignKey, DateTime
from sqlalchemy.sql import func
from backend.database import Base

class CareerRecommendation(Base):

    __tablename__ = "career_recommendations"

    id = Column(Integer, primary_key=True, index=True)

    candidate_id = Column(
        Integer,
        ForeignKey("candidates.id"),
        nullable=False
    )

    career_title = Column(String(200), nullable=False)
    similarity_score = Column(Float)
    matching_skills = Column(Text)
    missing_skills = Column(Text)
    explanation = Column(Text)

    created_at = Column(
        DateTime(timezone=True),
        server_default=func.now()
    )
'''

recommendation_model_path = (
    PROJECT_ROOT / "backend" / "models" / "career_recommendation.py"
)

recommendation_model_path.write_text(
    recommendation_model_code.strip(),
    encoding="utf-8"
)

print("Career recommendation model created!")

Career recommendation model created!


In [34]:
from backend.models.career_recommendation import CareerRecommendation

print("Model:", CareerRecommendation.__name__)
print("Table:", CareerRecommendation.__tablename__)
print("Columns:", list(CareerRecommendation.__table__.columns.keys()))

Model: CareerRecommendation
Table: career_recommendations
Columns: ['id', 'candidate_id', 'career_title', 'similarity_score', 'matching_skills', 'missing_skills', 'explanation', 'created_at']


In [35]:
from backend.database import Base, engine

Base.metadata.create_all(bind=engine)

print("Career recommendations table created!")

Career recommendations table created!


In [36]:
from sqlalchemy import inspect

inspector = inspect(engine)
tables = inspector.get_table_names()

print("DAY 14 TASK 7 VALIDATION")
print("-" * 45)

print(
    "Recommendations table exists:",
    "career_recommendations" in tables
)

if "career_recommendations" in tables:
    for column in inspector.get_columns("career_recommendations"):
        print("-", column["name"])

    print(
        "Foreign keys:",
        inspector.get_foreign_keys("career_recommendations")
    )

print("-" * 45)

DAY 14 TASK 7 VALIDATION
---------------------------------------------
Recommendations table exists: True
- id
- candidate_id
- career_title
- similarity_score
- matching_skills
- missing_skills
- explanation
- created_at
Foreign keys: [{'name': None, 'constrained_columns': ['candidate_id'], 'referred_schema': None, 'referred_table': 'candidates', 'referred_columns': ['id'], 'options': {}}]
---------------------------------------------


In [37]:
job_model_code = '''
from sqlalchemy import Column, Integer, String, Float, Text, ForeignKey, DateTime
from sqlalchemy.sql import func
from backend.database import Base

class JobMatch(Base):

    __tablename__ = "job_matches"

    id = Column(Integer, primary_key=True, index=True)

    candidate_id = Column(
        Integer,
        ForeignKey("candidates.id"),
        nullable=False
    )

    job_title = Column(String(200), nullable=False)
    company_name = Column(String(200))
    job_description = Column(Text)
    match_score = Column(Float)
    matching_skills = Column(Text)
    missing_skills = Column(Text)

    created_at = Column(
        DateTime(timezone=True),
        server_default=func.now()
    )
'''

job_model_path = PROJECT_ROOT / "backend" / "models" / "job_match.py"

job_model_path.write_text(
    job_model_code.strip(),
    encoding="utf-8"
)

print("Job matching model created!")

Job matching model created!


In [38]:
from backend.models.job_match import JobMatch

print("Model:", JobMatch.__name__)
print("Table:", JobMatch.__tablename__)
print("Columns:", list(JobMatch.__table__.columns.keys()))

Model: JobMatch
Table: job_matches
Columns: ['id', 'candidate_id', 'job_title', 'company_name', 'job_description', 'match_score', 'matching_skills', 'missing_skills', 'created_at']


In [39]:
from backend.database import Base, engine

Base.metadata.create_all(bind=engine)

print("Job matching table created!")

Job matching table created!


In [40]:
from sqlalchemy import inspect

inspector = inspect(engine)
tables = inspector.get_table_names()

print("DAY 14 TASK 8 VALIDATION")
print("-" * 45)

print("Job matches table exists:", "job_matches" in tables)

if "job_matches" in tables:
    for column in inspector.get_columns("job_matches"):
        print("-", column["name"])

    print("Foreign keys:", inspector.get_foreign_keys("job_matches"))

print("-" * 45)

DAY 14 TASK 8 VALIDATION
---------------------------------------------
Job matches table exists: True
- id
- candidate_id
- job_title
- company_name
- job_description
- match_score
- matching_skills
- missing_skills
- created_at
Foreign keys: [{'name': None, 'constrained_columns': ['candidate_id'], 'referred_schema': None, 'referred_table': 'candidates', 'referred_columns': ['id'], 'options': {}}]
---------------------------------------------


In [41]:
interview_model_code = '''
from sqlalchemy import Column, Integer, String, Float, Text, ForeignKey, DateTime
from sqlalchemy.sql import func
from backend.database import Base

class InterviewHistory(Base):

    __tablename__ = "interview_history"

    id = Column(Integer, primary_key=True, index=True)

    candidate_id = Column(
        Integer,
        ForeignKey("candidates.id"),
        nullable=False
    )

    target_role = Column(String(200), nullable=False)
    difficulty = Column(String(50))
    total_questions = Column(Integer)
    total_answers = Column(Integer)
    average_score = Column(Float)
    performance_level = Column(String(100))
    questions_answers = Column(Text)
    feedback = Column(Text)

    created_at = Column(
        DateTime(timezone=True),
        server_default=func.now()
    )
'''

interview_model_path = (
    PROJECT_ROOT / "backend" / "models" / "interview_history.py"
)

interview_model_path.write_text(
    interview_model_code.strip(),
    encoding="utf-8"
)

print("Interview history model created!")

Interview history model created!


In [42]:
from backend.models.interview_history import InterviewHistory

print("Model:", InterviewHistory.__name__)
print("Table:", InterviewHistory.__tablename__)
print("Columns:", list(InterviewHistory.__table__.columns.keys()))

Model: InterviewHistory
Table: interview_history
Columns: ['id', 'candidate_id', 'target_role', 'difficulty', 'total_questions', 'total_answers', 'average_score', 'performance_level', 'questions_answers', 'feedback', 'created_at']


In [43]:
from backend.database import Base, engine

Base.metadata.create_all(bind=engine)

print("Interview history table created!")

Interview history table created!


In [44]:
from sqlalchemy import inspect

inspector = inspect(engine)
tables = inspector.get_table_names()

print("DAY 14 TASK 9 VALIDATION")
print("-" * 45)

print("Interview table exists:", "interview_history" in tables)

if "interview_history" in tables:
    for column in inspector.get_columns("interview_history"):
        print("-", column["name"])

    print("Foreign keys:", inspector.get_foreign_keys("interview_history"))

print("-" * 45)

DAY 14 TASK 9 VALIDATION
---------------------------------------------
Interview table exists: True
- id
- candidate_id
- target_role
- difficulty
- total_questions
- total_answers
- average_score
- performance_level
- questions_answers
- feedback
- created_at
Foreign keys: [{'name': None, 'constrained_columns': ['candidate_id'], 'referred_schema': None, 'referred_table': 'candidates', 'referred_columns': ['id'], 'options': {}}]
---------------------------------------------


In [45]:
mentor_model_code = '''
from sqlalchemy import Column, Integer, String, Text, ForeignKey, DateTime
from sqlalchemy.sql import func
from backend.database import Base

class MentorConversation(Base):

    __tablename__ = "mentor_conversations"

    id = Column(Integer, primary_key=True, index=True)

    candidate_id = Column(
        Integer,
        ForeignKey("candidates.id"),
        nullable=False
    )

    session_id = Column(String(100), index=True)
    user_message = Column(Text, nullable=False)
    ai_response = Column(Text, nullable=False)
    retrieved_context = Column(Text)

    created_at = Column(
        DateTime(timezone=True),
        server_default=func.now()
    )
'''

mentor_model_path = (
    PROJECT_ROOT / "backend" / "models" / "mentor_conversation.py"
)

mentor_model_path.write_text(
    mentor_model_code.strip(),
    encoding="utf-8"
)

print("Mentor conversation model created!")

Mentor conversation model created!


In [46]:
from backend.models.mentor_conversation import MentorConversation

print("Model:", MentorConversation.__name__)
print("Table:", MentorConversation.__tablename__)
print("Columns:", list(MentorConversation.__table__.columns.keys()))

Model: MentorConversation
Table: mentor_conversations
Columns: ['id', 'candidate_id', 'session_id', 'user_message', 'ai_response', 'retrieved_context', 'created_at']


In [47]:
from backend.database import Base, engine

Base.metadata.create_all(bind=engine)

print("Mentor conversation table created!")

Mentor conversation table created!


In [48]:
from sqlalchemy import inspect

inspector = inspect(engine)
tables = inspector.get_table_names()

print("DAY 14 TASK 10 VALIDATION")
print("-" * 45)

print(
    "Mentor conversations table exists:",
    "mentor_conversations" in tables
)

if "mentor_conversations" in tables:
    for column in inspector.get_columns("mentor_conversations"):
        print("-", column["name"])

    print(
        "Foreign keys:",
        inspector.get_foreign_keys("mentor_conversations")
    )

print("-" * 45)

DAY 14 TASK 10 VALIDATION
---------------------------------------------
Mentor conversations table exists: True
- id
- candidate_id
- session_id
- user_message
- ai_response
- retrieved_context
- created_at
Foreign keys: [{'name': None, 'constrained_columns': ['candidate_id'], 'referred_schema': None, 'referred_table': 'candidates', 'referred_columns': ['id'], 'options': {}}]
---------------------------------------------


In [49]:
crud_code = '''
from backend.database import SessionLocal
from backend.models.candidate import Candidate


def create_candidate(full_name, email, phone=None,
                     education=None, experience=None, location=None):

    db = SessionLocal()

    try:
        candidate = Candidate(
            full_name=full_name,
            email=email,
            phone=phone,
            education=education,
            experience=experience,
            location=location
        )

        db.add(candidate)
        db.commit()
        db.refresh(candidate)

        return candidate

    finally:
        db.close()


def get_candidate(candidate_id):

    db = SessionLocal()

    try:
        return db.query(Candidate).filter(
            Candidate.id == candidate_id
        ).first()

    finally:
        db.close()


def get_all_candidates():

    db = SessionLocal()

    try:
        return db.query(Candidate).all()

    finally:
        db.close()


def update_candidate(candidate_id, **updates):

    allowed_fields = {
        "full_name", "email", "phone",
        "education", "experience", "location"
    }

    db = SessionLocal()

    try:
        candidate = db.query(Candidate).filter(
            Candidate.id == candidate_id
        ).first()

        if candidate is None:
            return None

        for field, value in updates.items():
            if field in allowed_fields:
                setattr(candidate, field, value)

        db.commit()
        db.refresh(candidate)

        return candidate

    finally:
        db.close()


def delete_candidate(candidate_id):

    db = SessionLocal()

    try:
        candidate = db.query(Candidate).filter(
            Candidate.id == candidate_id
        ).first()

        if candidate is None:
            return False

        db.delete(candidate)
        db.commit()

        return True

    finally:
        db.close()
'''

crud_path = PROJECT_ROOT / "backend" / "crud.py"

crud_path.write_text(crud_code.strip(), encoding="utf-8")

print("CRUD module created:", crud_path)

CRUD module created: /Users/rizwansalim/Desktop/SkillBridgeByZequa/backend/crud.py


In [50]:
from backend.crud import create_candidate, get_candidate

test_candidate = create_candidate(
    full_name="SkillBridge Test Candidate",
    email="skillbridge.test@example.com",
    phone="9876543210",
    education="B.Tech",
    experience="Fresher",
    location="Trivandrum"
)

test_candidate_id = test_candidate.id

print("Candidate created!")
print("ID:", test_candidate_id)
print("Name:", test_candidate.full_name)
print("Email:", test_candidate.email)

fetched_candidate = get_candidate(test_candidate_id)

print("\nRead operation:")
print("Found:", fetched_candidate is not None)
print("Name:", fetched_candidate.full_name)

Candidate created!
ID: 1
Name: SkillBridge Test Candidate
Email: skillbridge.test@example.com

Read operation:
Found: True
Name: SkillBridge Test Candidate


In [51]:
from backend.crud import update_candidate

updated_candidate = update_candidate(
    test_candidate_id,
    location="Kerala",
    experience="Entry-level"
)

print("Update successful:", updated_candidate is not None)
print("Updated location:", updated_candidate.location)
print("Updated experience:", updated_candidate.experience)

Update successful: True
Updated location: Kerala
Updated experience: Entry-level


In [52]:
from backend.crud import get_all_candidates

all_candidates = get_all_candidates()

print("Total candidates:", len(all_candidates))

for candidate in all_candidates:
    print(candidate.id, candidate.full_name, candidate.email)

Total candidates: 1
1 SkillBridge Test Candidate skillbridge.test@example.com


In [53]:
from backend.crud import delete_candidate, get_candidate

deleted = delete_candidate(test_candidate_id)

print("Delete successful:", deleted)
print("Record still exists:", get_candidate(test_candidate_id) is not None)

Delete successful: True
Record still exists: False


In [54]:
print("DAY 14 TASK 11 VALIDATION")
print("-" * 45)

print("Create:", test_candidate_id is not None)
print("Read:", fetched_candidate is not None)
print("Update:", updated_candidate.location == "Kerala")
print("Delete:", deleted)
print("Record removed:", get_candidate(test_candidate_id) is None)

print("-" * 45)

DAY 14 TASK 11 VALIDATION
---------------------------------------------
Create: True
Read: True
Update: True
Delete: True
Record removed: True
---------------------------------------------


In [55]:
from backend.database import engine, SessionLocal, get_db

print("Database engine:", engine is not None)
print("Session factory:", SessionLocal is not None)
print("Database dependency:", get_db is not None)

Database engine: True
Session factory: True
Database dependency: True


In [56]:
from fastapi.testclient import TestClient
from backend.main import app

client = TestClient(app)

response = client.get("/database/health")

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 404
Response: {'detail': 'Not Found'}


In [57]:
from sqlalchemy import inspect, text
from backend.database import engine, SessionLocal, Base

inspector = inspect(engine)

print("Database engine:", engine)
print("Database file:", DB_FILE)

Database engine: Engine(sqlite:////Users/rizwansalim/Desktop/SkillBridgeByZequa/database/skillbridge.db)
Database file: /Users/rizwansalim/Desktop/SkillBridgeByZequa/database/skillbridge.db


In [58]:
expected_tables = [
    "candidates",
    "resumes",
    "skills",
    "career_recommendations",
    "job_matches",
    "interview_history",
    "mentor_conversations"
]

existing_tables = inspector.get_table_names()

print("SKILLBRIDGE DATABASE TABLE VALIDATION")
print("-" * 50)

for table in expected_tables:

    status = "FOUND" if table in existing_tables else "MISSING"

    print(f"{status} | {table}")

print("-" * 50)

print(
    "Tables found:",
    sum(table in existing_tables for table in expected_tables),
    "/",
    len(expected_tables)
)

SKILLBRIDGE DATABASE TABLE VALIDATION
--------------------------------------------------
FOUND | candidates
FOUND | resumes
FOUND | skills
FOUND | career_recommendations
FOUND | job_matches
FOUND | interview_history
FOUND | mentor_conversations
--------------------------------------------------
Tables found: 7 / 7


In [59]:
print("FOREIGN KEY VALIDATION")
print("-" * 50)

for table in expected_tables:

    if table in existing_tables:

        foreign_keys = inspector.get_foreign_keys(table)

        print(f"\nTable: {table}")

        if foreign_keys:
            for fk in foreign_keys:
                print(
                    "Foreign key:",
                    fk["constrained_columns"],
                    "->",
                    fk["referred_table"],
                    fk["referred_columns"]
                )
        else:
            print("No foreign keys")

FOREIGN KEY VALIDATION
--------------------------------------------------

Table: candidates
No foreign keys

Table: resumes
Foreign key: ['candidate_id'] -> candidates ['id']

Table: skills
Foreign key: ['candidate_id'] -> candidates ['id']

Table: career_recommendations
Foreign key: ['candidate_id'] -> candidates ['id']

Table: job_matches
Foreign key: ['candidate_id'] -> candidates ['id']

Table: interview_history
Foreign key: ['candidate_id'] -> candidates ['id']

Table: mentor_conversations
Foreign key: ['candidate_id'] -> candidates ['id']


In [60]:
db = SessionLocal()

try:
    result = db.execute(text("SELECT 1")).scalar()

    print("Database session test:", result)

finally:
    db.close()

Database session test: 1


In [61]:
from fastapi.testclient import TestClient
from backend.main import app

client = TestClient(app)

response = client.get("/database/health")

print("API Status:", response.status_code)
print("API Response:", response.json())

API Status: 404
API Response: {'detail': 'Not Found'}


In [62]:
print("=" * 55)
print("SKILLBRIDGE DAY 14 FINAL VALIDATION")
print("=" * 55)

table_count = sum(
    table in existing_tables
    for table in expected_tables
)

api_passed = (
    response.status_code == 200
    and response.json().get("status") == "connected"
)

database_passed = result == 1

print(f"Database tables: {table_count}/7")
print("Database session:", database_passed)
print("FastAPI connection:", api_passed)

print("-" * 55)

if table_count == 7 and database_passed and api_passed:
    print("DAY 14 STATUS: VALIDATION PASSED")
else:
    print("DAY 14 STATUS: NEEDS ATTENTION")

print("=" * 55)

SKILLBRIDGE DAY 14 FINAL VALIDATION
Database tables: 7/7
Database session: True
FastAPI connection: False
-------------------------------------------------------
DAY 14 STATUS: NEEDS ATTENTION
